In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical
import gymnasium as gymnasium

# 1. Definición de la Red Actor-Crítico (Red Compartida con dos cabezales)
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()

        # Tronco común para extraer características
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, action_dim) # Salida: logits para probabilidades de acciones discretas
        )

        self.critic = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)          # Salida: Valor V(s) esperado
        )

    def forward(self, state):
        return self.actor(state), self.critic(state)

    def get_action(self, state):
        logits, value = self.forward(state)
        dist = Categorical(logits=logits)
        action = dist.sample()
        return action.item(), dist.log_prob(action), value

# 2. Función para calcular GAE (Generalized Advantage Estimation) y Retornos
def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0
    returns = []

    # Recorrer en reversa las trayectorias del rollout
    for step in reversed(range(len(rewards))):
        if step == len(rewards) - 1:
            next_non_terminal = 1.0 - dones[step]
            next_val = next_value
        else:
            next_non_terminal = 1.0 - dones[step]
            next_val = values[step + 1]

        delta = rewards[step] + gamma * next_val * next_non_terminal - values[step]
        gae = delta + gamma * lam * next_non_terminal * gae
        advantages.insert(0, gae)
        returns.insert(0, gae + values[step])

    return torch.tensor(advantages, dtype=torch.float32), torch.tensor(returns, dtype=torch.float32)

# 3. Bucle Principal de Entrenamiento PPO
def train_ppo():
    env = gymnasium.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    model = ActorCritic(state_dim, action_dim)
    optimizer = optim.Adam(model.parameters(), lr=3e-4)

    episodes = 200
    gamma = 0.99
    lam = 0.95
    clip_eps = 0.2
    epochs_ppo = 4

    for episode in range(episodes):
        state, _ = env.reset()

        # Buffers temporales para el rollout actual
        states, actions, log_probs_old, rewards, values, dones = [], [], [], [], [], []

        done = False
        score = 0

        while not done:
            state_t = torch.tensor(state, dtype=torch.float32)
            action, log_prob, value = model.get_action(state_t)

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            states.append(state_t)
            actions.append(action)
            log_probs_old.append(log_prob)
            rewards.append(reward)
            values.append(value.item())
            dones.append(done)

            state = next_state
            score += reward

        # Valor del estado final para bootstrapping en GAE
        with torch.no_grad():
            _, next_value = model.forward(torch.tensor(state, dtype=torch.float32))
            next_value = next_value.item()

        # Calcular Ventajas y Retornos
        advantages, returns = compute_gae(rewards, values, dones, next_value, gamma, lam)
        # Normalizar ventajas para estabilidad numérica
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        states = torch.stack(states)
        actions = torch.tensor(actions)
        log_probs_old = torch.stack(log_probs_old)

        # Actualización PPO por múltiples épocas sobre los mismos datos
        for _ in range(epochs_ppo):
            logits, values_pred = model(states)
            dist = Categorical(logits=logits)
            log_probs = dist.log_prob(actions)
            entropy = dist.entropy().mean()

            # Ratio de probabilidad: r_t(theta)
            ratios = torch.exp(log_probs - log_probs_old.detach())

            # Surrogate Objective con Clipping
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1.0 - clip_eps, 1.0 + clip_eps) * advantages
            actor_loss = -torch.min(surr1, surr2).mean()

            # Pérdida del Crítico (Error cuadrático medio)
            critic_loss = nn.MSELoss()(values_pred.squeeze(), returns)

            # Pérdida Total (incluyendo bono de entropía para fomentar exploración)
            loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Episodio {episode+1} | Puntaje: {score}")

    env.close()

if __name__ == "__main__":
    train_ppo()

Episodio 1 | Puntaje: 18.0
Episodio 2 | Puntaje: 15.0
Episodio 3 | Puntaje: 22.0
Episodio 4 | Puntaje: 59.0
Episodio 5 | Puntaje: 16.0
Episodio 6 | Puntaje: 32.0
Episodio 7 | Puntaje: 41.0
Episodio 8 | Puntaje: 20.0
Episodio 9 | Puntaje: 23.0
Episodio 10 | Puntaje: 32.0
Episodio 11 | Puntaje: 17.0
Episodio 12 | Puntaje: 24.0
Episodio 13 | Puntaje: 72.0
Episodio 14 | Puntaje: 14.0
Episodio 15 | Puntaje: 48.0
Episodio 16 | Puntaje: 37.0
Episodio 17 | Puntaje: 19.0
Episodio 18 | Puntaje: 26.0
Episodio 19 | Puntaje: 13.0
Episodio 20 | Puntaje: 54.0
Episodio 21 | Puntaje: 41.0
Episodio 22 | Puntaje: 11.0
Episodio 23 | Puntaje: 14.0
Episodio 24 | Puntaje: 16.0
Episodio 25 | Puntaje: 20.0
Episodio 26 | Puntaje: 15.0
Episodio 27 | Puntaje: 16.0
Episodio 28 | Puntaje: 11.0
Episodio 29 | Puntaje: 11.0
Episodio 30 | Puntaje: 19.0
Episodio 31 | Puntaje: 15.0
Episodio 32 | Puntaje: 11.0
Episodio 33 | Puntaje: 11.0
Episodio 34 | Puntaje: 27.0
Episodio 35 | Puntaje: 20.0
Episodio 36 | Puntaje: 13.0
E